# WM-811K Wafer Defect Classification — CNN Pipeline

**Goal:** Train a CNN to classify wafer maps into 9 classes (8 defect types + "none").  
**Principles:** No data leakage · Stratified splits · Augmentation on train only · Early stopping · Class-weighted loss.

## 1. Imports & Configuration

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, confusion_matrix
from sklearn.preprocessing import LabelEncoder

@dataclass
class CFG:
    seed: int = 42
    batch_size: int = 64
    lr: float = 1e-3
    max_epochs: int = 100
    es_patience: int = 10      # early stopping patience
    lr_patience: int = 5       # ReduceLROnPlateau patience
    lr_factor: float = 0.5
    val_frac: float = 0.20
    num_workers: int = 2
    checkpoint: str = '../best_model.pt'
    figures_dir: str = '../figures'
    tables_dir: str = '../tables'

cfg = CFG()

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(cfg.seed)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
Path(cfg.figures_dir).mkdir(parents=True, exist_ok=True)
Path(cfg.tables_dir).mkdir(parents=True, exist_ok=True)

## 2. Data Loading & Preparation

We reuse the unpacking pattern from the EDA notebook to unpack the nested arrays in `lotName`, `trainTestLabel`, and `failureType`.

In [ ]:
df = pd.read_pickle('../data/LSWMD_modern.pkl')

# Fix typo from original dataset
df.rename(columns={'trianTestLabel': 'trainTestLabel'}, inplace=True)

def unpack(x):
    arr = np.asarray(x)
    if arr.size == 0:
        return None
    return arr[0][0]

for col in ['lotName', 'trainTestLabel', 'failureType']:
    df[col] = df[col].apply(unpack)

# Keep only labeled rows with a known failure type
labeled = df[df['trainTestLabel'].notna() & df['failureType'].notna()].copy()
print(f'Labeled rows: {len(labeled):,}')
print(labeled['failureType'].value_counts())

In [ ]:
# Encode failure type to integer labels — deterministic alphabetical order
le = LabelEncoder()
labeled['label'] = le.fit_transform(labeled['failureType'])
CLASS_NAMES = list(le.classes_)
N_CLASSES = len(CLASS_NAMES)
print(f'Classes ({N_CLASSES}):', CLASS_NAMES)

### Class distribution

Visualising imbalance before the split — this informs our choice of class-weighted loss.

In [ ]:
counts = labeled['failureType'].value_counts().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(x=counts.index, y=counts.values, palette='viridis', ax=ax)
ax.set_title('Class distribution — labeled wafers')
ax.set_xlabel('Failure type')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=30)
for bar, val in zip(ax.patches, counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 200,
            f'{val:,}', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.savefig(f'{cfg.figures_dir}/class_distribution.png', dpi=150)
plt.show()

## 3. Train / Val / Test Split (no leakage)

The dataset ships with a `trainTestLabel` column. We use it as the primary split.  
Val is carved from the training pool using a stratified shuffle — every class is represented proportionally.

In [ ]:
train_pool = labeled[labeled['trainTestLabel'] == 'Training'].copy()
test_df    = labeled[labeled['trainTestLabel'] == 'Test'].copy()

sss = StratifiedShuffleSplit(n_splits=1, test_size=cfg.val_frac, random_state=cfg.seed)
train_idx, val_idx = next(sss.split(train_pool, train_pool['label']))

train_df = train_pool.iloc[train_idx].copy()
val_df   = train_pool.iloc[val_idx].copy()

print(f'Train : {len(train_df):,}')
print(f'Val   : {len(val_df):,}')
print(f'Test  : {len(test_df):,}')

In [ ]:
# Compute mean and std from TRAINING pixels only — no test/val statistics used
train_maps = np.stack([np.asarray(w, dtype=np.float32) for w in train_df['waferMap']])
PIXEL_MEAN = float(train_maps.mean())
PIXEL_STD  = float(train_maps.std()) + 1e-8  # guard against zero std
print(f'Training pixel mean: {PIXEL_MEAN:.4f}, std: {PIXEL_STD:.4f}')

## 4. PyTorch Dataset & Transforms

Augmentations applied **only** to the training split.  
All splits share the same normalization (computed from training data).

In [ ]:
class WaferMapDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.maps   = [np.asarray(w, dtype=np.float32) for w in dataframe['waferMap']]
        self.labels = dataframe['label'].values
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        # Shape: (H, W) → (1, H, W)
        img = torch.from_numpy(self.maps[idx]).unsqueeze(0)
        if self.transform:
            img = self.transform(img)
        return img, int(self.labels[idx])


normalize = T.Normalize(mean=[PIXEL_MEAN], std=[PIXEL_STD])

train_transform = T.Compose([
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomApply([T.RandomRotation(degrees=(90, 90))], p=0.5),
    normalize,
])

eval_transform = T.Compose([normalize])

train_ds = WaferMapDataset(train_df, transform=train_transform)
val_ds   = WaferMapDataset(val_df,   transform=eval_transform)
test_ds  = WaferMapDataset(test_df,  transform=eval_transform)

print(f'Dataset sizes — train: {len(train_ds)}, val: {len(val_ds)}, test: {len(test_ds)}')

## 5. Class Weights & DataLoaders

In [ ]:
# Inverse-frequency weights — computed from training set only
label_counts = np.bincount(train_df['label'].values, minlength=N_CLASSES).astype(float)
class_weights = torch.tensor(
    len(train_df) / (N_CLASSES * label_counts), dtype=torch.float32
).to(DEVICE)

print('Class weights:')
for name, w in zip(CLASS_NAMES, class_weights.cpu().numpy()):
    print(f'  {name:<12} {w:.3f}')

In [ ]:
train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                          num_workers=cfg.num_workers, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=cfg.batch_size, shuffle=False,
                          num_workers=cfg.num_workers, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=cfg.batch_size, shuffle=False,
                          num_workers=cfg.num_workers, pin_memory=True)

## 6. CNN Architecture

Three conv blocks halve spatial dimensions at each stage.  
Input `(1, 45, 48)` → `(32, 22, 24)` → `(64, 11, 12)` → `(128, 5, 6)` → flatten 3,840 → FC256 → FC9.

In [ ]:
def conv_block(in_ch, out_ch):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(2, 2),
    )

class WaferCNN(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.features = nn.Sequential(
            conv_block(1, 32),
            conv_block(32, 64),
            conv_block(64, 128),
        )
        # Compute flattened size dynamically — safe against rounding in MaxPool
        with torch.no_grad():
            dummy = torch.zeros(1, 1, 45, 48)
            flat_size = self.features(dummy).numel()

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat_size, 256, bias=False),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, n_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


model = WaferCNN(N_CLASSES).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f'\nTrainable parameters: {total_params:,}')

## 7. Loss, Optimizer, Scheduler & Early Stopping

In [ ]:
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=cfg.lr)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=cfg.lr_patience, factor=cfg.lr_factor, verbose=True
)

class EarlyStopping:
    def __init__(self, patience, checkpoint_path):
        self.patience = patience
        self.path = checkpoint_path
        self.best_loss = float('inf')
        self.counter = 0
        self.stop = False

    def __call__(self, val_loss, model):
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.counter = 0
            torch.save(model.state_dict(), self.path)
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True

early_stop = EarlyStopping(cfg.es_patience, cfg.checkpoint)

## 8. Training Loop

In [ ]:
def run_epoch(loader, model, criterion, optimizer=None):
    """One forward pass over loader. Pass optimizer=None for eval mode."""
    training = optimizer is not None
    model.train() if training else model.eval()
    total_loss, correct, total = 0.0, 0, 0

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            logits = model(imgs)
            loss = criterion(logits, labels)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(labels)
            correct += (logits.argmax(1) == labels).sum().item()
            total += len(labels)

    return total_loss / total, correct / total


history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

print(f'{"Epoch":>6}  {"Train Loss":>10}  {"Train Acc":>10}  {"Val Loss":>10}  {"Val Acc":>10}  {"LR":>8}')
print('-' * 65)

for epoch in range(1, cfg.max_epochs + 1):
    tr_loss, tr_acc = run_epoch(train_loader, model, criterion, optimizer)
    vl_loss, vl_acc = run_epoch(val_loader,   model, criterion)

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(vl_acc)

    current_lr = optimizer.param_groups[0]['lr']
    print(f'{epoch:>6}  {tr_loss:>10.4f}  {tr_acc:>10.4f}  {vl_loss:>10.4f}  {vl_acc:>10.4f}  {current_lr:>8.2e}')

    scheduler.step(vl_loss)
    early_stop(vl_loss, model)
    if early_stop.stop:
        print(f'\nEarly stopping at epoch {epoch}. Best val loss: {early_stop.best_loss:.4f}')
        break

print('Training complete.')

## 9. Training Curves

In [ ]:
epochs_ran = range(1, len(history['train_loss']) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs_ran, history['train_loss'], label='Train')
ax1.plot(epochs_ran, history['val_loss'],   label='Val')
ax1.set_title('Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Cross-entropy loss')
ax1.legend()

ax2.plot(epochs_ran, history['train_acc'], label='Train')
ax2.plot(epochs_ran, history['val_acc'],   label='Val')
ax2.set_title('Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.legend()

plt.tight_layout()
plt.savefig(f'{cfg.figures_dir}/training_curves.png', dpi=150)
plt.show()

## 10. Test Set Evaluation

Load the best checkpoint saved by EarlyStopping. Test set is touched **only here**.

In [ ]:
model.load_state_dict(torch.load(cfg.checkpoint, map_location=DEVICE))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(DEVICE)
        preds = model(imgs).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

print('=== Test Set Results ===')
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES, digits=4))

## 11. Confusion Matrix

In [ ]:
cm = confusion_matrix(all_labels, all_preds, normalize='true')

fig, ax = plt.subplots(figsize=(9, 7))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
disp.plot(ax=ax, colorbar=True, cmap='Blues', values_format='.2f')
ax.set_title('Confusion matrix (normalised by true label)')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.savefig(f'{cfg.figures_dir}/confusion_matrix.png', dpi=150)
plt.show()

## 12. Baseline Results

Lock in all baseline metrics **before** any architecture or augmentation changes. This section is the reference point every subsequent experiment is measured against.

In [ ]:
def evaluate_model(model, loader, class_names, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            preds = model(imgs).argmax(1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)
    report = classification_report(all_labels, all_preds, target_names=class_names, output_dict=True)
    summary = {
        'accuracy':    report['accuracy'],
        'macro_f1':    report['macro avg']['f1-score'],
        'weighted_f1': report['weighted avg']['f1-score'],
    }
    return all_preds, all_labels, report, summary


def report_to_markdown(report, class_names):
    rows = ['| Class | Precision | Recall | F1-score | Support |',
            '|-------|-----------|--------|----------|---------|']
    for cls in class_names:
        r = report[cls]
        rows.append(f"| {cls} | {r['precision']:.4f} | {r['recall']:.4f} | {r['f1-score']:.4f} | {int(r['support'])} |")
    rows.append(f"| **accuracy** | | | {report['accuracy']:.4f} | |")
    rows.append(f"| **macro avg** | {report['macro avg']['precision']:.4f} | {report['macro avg']['recall']:.4f} | {report['macro avg']['f1-score']:.4f} | {int(report['macro avg']['support'])} |")
    rows.append(f"| **weighted avg** | {report['weighted avg']['precision']:.4f} | {report['weighted avg']['recall']:.4f} | {report['weighted avg']['f1-score']:.4f} | {int(report['weighted avg']['support'])} |")
    return '\n'.join(rows)


def comparison_to_markdown(rows):
    lines = ['| Run | Accuracy | Macro F1 | Weighted F1 | Epochs |',
             '|-----|----------|----------|-------------|--------|']
    for r in rows:
        lines.append(f"| {r['run']} | {r['accuracy']:.4f} | {r['macro_f1']:.4f} | {r['weighted_f1']:.4f} | {r['epochs']} |")
    return '\n'.join(lines)


CMAP_WM = plt.cm.get_cmap('RdYlGn_r', 3)  # shared colormap for wafer map displays

In [ ]:
# Load best baseline checkpoint and run inference on test set
model.load_state_dict(torch.load(cfg.checkpoint, map_location=DEVICE))
baseline_preds, baseline_labels, baseline_report, baseline_summary = evaluate_model(
    model, test_loader, CLASS_NAMES, DEVICE
)
baseline_summary['epochs'] = len(history['train_loss'])

# Per-class F1 horizontal bar chart
f1_scores = {cls: baseline_report[cls]['f1-score'] for cls in CLASS_NAMES}
sorted_cls = sorted(f1_scores, key=f1_scores.get)

fig, ax = plt.subplots(figsize=(8, 5))
bar_colors = ['#d62728' if f1_scores[c] < 0.5 else '#2ca02c' for c in sorted_cls]
ax.barh(sorted_cls, [f1_scores[c] for c in sorted_cls], color=bar_colors)
ax.set_xlabel('F1-score')
ax.set_title('Per-class F1 — Baseline CNN')
ax.set_xlim(0, 1)
ax.axvline(baseline_summary['macro_f1'], color='navy', linestyle='--',
           label=f"Macro F1 = {baseline_summary['macro_f1']:.3f}")
ax.legend()
plt.tight_layout()
plt.savefig(f'{cfg.figures_dir}/per_class_f1_baseline.png', dpi=150)
plt.show()

In [ ]:
# 4x4 sample predictions grid — green border = correct, red = wrong
rng_sample = np.random.default_rng(cfg.seed)
sample_idx = rng_sample.choice(len(test_ds), size=16, replace=False)

fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for ax, idx in zip(axes.flatten(), sample_idx):
    img, true_label = test_ds[int(idx)]
    pred_label = int(baseline_preds[idx])
    wmap = img.squeeze().numpy() * PIXEL_STD + PIXEL_MEAN
    ax.imshow(wmap, cmap=CMAP_WM, vmin=0, vmax=2, interpolation='nearest')
    correct = (pred_label == true_label)
    color = 'green' if correct else 'red'
    ax.set_title(f'T: {CLASS_NAMES[true_label]}\nP: {CLASS_NAMES[pred_label]}',
                 fontsize=7, color=color)
    for spine in ax.spines.values():
        spine.set_edgecolor(color)
        spine.set_linewidth(3)
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

fig.suptitle('Sample Predictions — Baseline CNN  (green = correct, red = wrong)', fontsize=12)
plt.tight_layout()
plt.savefig(f'{cfg.figures_dir}/sample_predictions_baseline.png', dpi=150)
plt.show()

In [ ]:
# Save classification report as markdown
Path(f'{cfg.tables_dir}/classification_report_baseline.md').write_text(
    '# Baseline CNN — Classification Report\n\n' + report_to_markdown(baseline_report, CLASS_NAMES) + '\n'
)
print('Saved: tables/classification_report_baseline.md')

# Initialise model comparison table (overwritten fresh each time baseline runs)
comparison_rows = [dict(
    run='Baseline CNN',
    accuracy=baseline_summary['accuracy'],
    macro_f1=baseline_summary['macro_f1'],
    weighted_f1=baseline_summary['weighted_f1'],
    epochs=baseline_summary['epochs'],
)]
Path(f'{cfg.tables_dir}/model_comparison.md').write_text(
    '# Model Comparison\n\n' + comparison_to_markdown(comparison_rows) + '\n'
)
print('Saved: tables/model_comparison.md')
print(f"\nBaseline  Acc: {baseline_summary['accuracy']:.4f}  Macro F1: {baseline_summary['macro_f1']:.4f}  Weighted F1: {baseline_summary['weighted_f1']:.4f}")

## 13. Experiment: Enhanced Augmentation (RandomErasing)

The baseline is locked in above. We now add `RandomErasing` to the training transform and re-train from scratch.

`RandomErasing` randomly blanks a small rectangular patch of the wafer map at each training step. This forces the model not to over-rely on any single spatial region — useful for distinguishing `Edge-Loc` from `Loc`, which differ primarily by *where* on the wafer the defect cluster appears. It also acts as a regulariser against memorising specific pixel configurations.

Val and test datasets are **unchanged** — only the training augmentation differs.

In [ ]:
train_transform_v2 = T.Compose([
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomApply([T.RandomRotation(degrees=(90, 90))], p=0.5),
    normalize,
    T.RandomErasing(p=0.3, scale=(0.02, 0.15)),
])

train_ds_v2   = WaferMapDataset(train_df, transform=train_transform_v2)
train_loader_v2 = DataLoader(train_ds_v2, batch_size=cfg.batch_size, shuffle=True,
                             num_workers=cfg.num_workers, pin_memory=True)
print('Augmented train dataset ready:', len(train_ds_v2), 'samples')

In [ ]:
seed_everything(cfg.seed)
model_v2     = WaferCNN(N_CLASSES).to(DEVICE)
optimizer_v2 = optim.Adam(model_v2.parameters(), lr=cfg.lr)
scheduler_v2 = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_v2, mode='min', patience=cfg.lr_patience, factor=cfg.lr_factor, verbose=True
)
checkpoint_v2  = '../best_model_v2.pt'
early_stop_v2  = EarlyStopping(cfg.es_patience, checkpoint_v2)
history_v2     = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

print(f'{"Epoch":>6}  {"Train Loss":>10}  {"Train Acc":>10}  {"Val Loss":>10}  {"Val Acc":>10}  {"LR":>8}')
print('-' * 65)

for epoch in range(1, cfg.max_epochs + 1):
    tr_loss, tr_acc = run_epoch(train_loader_v2, model_v2, criterion, optimizer_v2)
    vl_loss, vl_acc = run_epoch(val_loader,      model_v2, criterion)

    history_v2['train_loss'].append(tr_loss)
    history_v2['val_loss'].append(vl_loss)
    history_v2['train_acc'].append(tr_acc)
    history_v2['val_acc'].append(vl_acc)

    current_lr = optimizer_v2.param_groups[0]['lr']
    print(f'{epoch:>6}  {tr_loss:>10.4f}  {tr_acc:>10.4f}  {vl_loss:>10.4f}  {vl_acc:>10.4f}  {current_lr:>8.2e}')

    scheduler_v2.step(vl_loss)
    early_stop_v2(vl_loss, model_v2)
    if early_stop_v2.stop:
        print(f'\nEarly stopping at epoch {epoch}. Best val loss: {early_stop_v2.best_loss:.4f}')
        break

print('Training complete.')

In [ ]:
epochs_v2 = range(1, len(history_v2['train_loss']) + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(epochs_v2, history_v2['train_loss'], label='Train')
ax1.plot(epochs_v2, history_v2['val_loss'],   label='Val')
ax1.set_title('Loss — Augmented CNN'); ax1.set_xlabel('Epoch'); ax1.legend()
ax2.plot(epochs_v2, history_v2['train_acc'], label='Train')
ax2.plot(epochs_v2, history_v2['val_acc'],   label='Val')
ax2.set_title('Accuracy — Augmented CNN'); ax2.set_xlabel('Epoch'); ax2.legend()
plt.tight_layout()
plt.savefig(f'{cfg.figures_dir}/training_curves_augmented.png', dpi=150)
plt.show()

In [ ]:
model_v2.load_state_dict(torch.load(checkpoint_v2, map_location=DEVICE))
aug_preds, aug_labels, aug_report, aug_summary = evaluate_model(
    model_v2, test_loader, CLASS_NAMES, DEVICE
)
aug_summary['epochs'] = len(history_v2['train_loss'])

# Per-class F1
f1_v2 = {cls: aug_report[cls]['f1-score'] for cls in CLASS_NAMES}
sorted_v2 = sorted(f1_v2, key=f1_v2.get)
fig, ax = plt.subplots(figsize=(8, 5))
bar_colors_v2 = ['#d62728' if f1_v2[c] < 0.5 else '#2ca02c' for c in sorted_v2]
ax.barh(sorted_v2, [f1_v2[c] for c in sorted_v2], color=bar_colors_v2)
ax.set_xlabel('F1-score')
ax.set_title('Per-class F1 — Augmented CNN (+ RandomErasing)')
ax.set_xlim(0, 1)
ax.axvline(aug_summary['macro_f1'], color='navy', linestyle='--',
           label=f"Macro F1 = {aug_summary['macro_f1']:.3f}")
ax.legend()
plt.tight_layout()
plt.savefig(f'{cfg.figures_dir}/per_class_f1_augmented.png', dpi=150)
plt.show()

# Sample predictions
sample_idx_v2 = np.random.default_rng(cfg.seed + 1).choice(len(test_ds), size=16, replace=False)
fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for ax, idx in zip(axes.flatten(), sample_idx_v2):
    img, true_label = test_ds[int(idx)]
    pred_label = int(aug_preds[idx])
    wmap = img.squeeze().numpy() * PIXEL_STD + PIXEL_MEAN
    ax.imshow(wmap, cmap=CMAP_WM, vmin=0, vmax=2, interpolation='nearest')
    correct = (pred_label == true_label)
    color = 'green' if correct else 'red'
    ax.set_title(f'T: {CLASS_NAMES[true_label]}\nP: {CLASS_NAMES[pred_label]}', fontsize=7, color=color)
    for spine in ax.spines.values():
        spine.set_edgecolor(color); spine.set_linewidth(3)
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
fig.suptitle('Sample Predictions — Augmented CNN  (green = correct, red = wrong)', fontsize=12)
plt.tight_layout()
plt.savefig(f'{cfg.figures_dir}/sample_predictions_augmented.png', dpi=150)
plt.show()

# Save tables
Path(f'{cfg.tables_dir}/classification_report_augmented.md').write_text(
    '# Augmented CNN — Classification Report\n\n' + report_to_markdown(aug_report, CLASS_NAMES) + '\n'
)
print('Saved: tables/classification_report_augmented.md')

comparison_rows.append(dict(
    run='Augmented CNN (+ RandomErasing)',
    accuracy=aug_summary['accuracy'],
    macro_f1=aug_summary['macro_f1'],
    weighted_f1=aug_summary['weighted_f1'],
    epochs=aug_summary['epochs'],
))
Path(f'{cfg.tables_dir}/model_comparison.md').write_text(
    '# Model Comparison\n\n' + comparison_to_markdown(comparison_rows) + '\n'
)
print('Saved: tables/model_comparison.md')
print(f"\nAugmented  Acc: {aug_summary['accuracy']:.4f}  Macro F1: {aug_summary['macro_f1']:.4f}  Weighted F1: {aug_summary['weighted_f1']:.4f}")

## 14. Model Comparison

Side-by-side view of all experiments. `tables/model_comparison.md` is the persistent record — add more rows as you try new architectures or training strategies, then re-run this cell.

In [ ]:
comp_text = Path(f'{cfg.tables_dir}/model_comparison.md').read_text()
# Parse markdown table rows (skip header and separator lines)
data_lines = [l.strip() for l in comp_text.split('\n')
              if l.startswith('|') and '---' not in l and 'Run' not in l]

runs, accs, macro_f1s, weighted_f1s = [], [], [], []
for line in data_lines:
    parts = [p.strip() for p in line.strip('|').split('|')]
    if len(parts) < 4:
        continue
    runs.append(parts[0])
    accs.append(float(parts[1]))
    macro_f1s.append(float(parts[2]))
    weighted_f1s.append(float(parts[3]))

x = np.arange(len(runs))
width = 0.25
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width, accs,         width, label='Accuracy',    color='steelblue')
ax.bar(x,         macro_f1s,    width, label='Macro F1',    color='darkorange')
ax.bar(x + width, weighted_f1s, width, label='Weighted F1', color='seagreen')
ax.set_xticks(x)
ax.set_xticklabels(runs, rotation=15, ha='right')
ax.set_ylim(0, 1)
ax.set_ylabel('Score')
ax.set_title('Model Comparison — Test Set Performance')
ax.legend()
ax.yaxis.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig(f'{cfg.figures_dir}/model_comparison.png', dpi=150)
plt.show()

print('\nFull comparison table:')
print(comp_text)